# Phase 30 -- compact prompt + full model re-review

Two changes bundled into one decision, per the user's explicit framing: the current
22-rule `SYSTEM_PROMPT` was written and tuned entirely against `gpt-4o-mini`'s
weaknesses (Phase 20-28), so testing a stronger candidate model with that same prompt
would still be paying its verbosity cost on every call without knowing whether a better
model needs it. This notebook:

1. Compacts `SYSTEM_PROMPT` (34% shorter by token count, no rule dropped) -- section 1.
2. Smoke-tests the compact prompt on a few known-tricky cases before spending on the
   full run -- section 2.
3. Runs the real decision-grade comparison: production as it ships today
   (`gpt-4o-mini`, **original** prompt -- the real baseline, unchanged) vs.
   `gemini-2.5-flash` and `gpt-4o` (**compact** prompt), all 60 `eval_v3` questions,
   paired identical context, regex proxy + Sonnet-5 judge -- same methodology as every
   past model decision in this project (Phase 7, 13, 15, 16, 19).

**Scope caveat, stated up front:** this design tests "new model + compact prompt" as
one bundled candidate against "old model + old prompt" as the baseline. It does **not**
isolate whether a win (if any) comes from the model or the prompt -- that would need a
4th arm (new model + original prompt) which was cut for cost. If a candidate wins, the
honest claim is "this configuration beats production," not "the model alone is better."

**Decision rule, fixed before seeing results** (Phase 13's discipline: state it before
looking, since deciding after triples the risk of just describing the noise): adopt a
candidate only if it beats production on **both** mean faithfulness and mean relevancy,
paired per question, ties excluded, by a real margin -- not simply a higher mean.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv
load_dotenv(PROJECT / ".env")

from openai import OpenAI
from src.generation import (OpenAICompatibleGenerator, PRODUCTION_API_KEY_ENV, PRODUCTION_BASE_URL,
                             PRODUCTION_MODEL_ID, CONTEXT_SYSTEM_PROMPT, SYSTEM_PROMPT,
                             extract_cited_articles, grounded_article_numbers, _format_context)
from src.retrieval import Retriever

EVAL_DIR = PROJECT / "eval"
print("original SYSTEM_PROMPT:", len(SYSTEM_PROMPT), "chars")


original SYSTEM_PROMPT: 7913 chars


## 1. Compact prompt

Same 22 rules, no rule dropped, phrasing tightened. Rules 14-20 (the calculator-block
rules) were kept conservative -- those are tied to real measured bugs from Phase 24-26
and are not worth risking for a marginal token saving. Rule 21 was tightened to say
"lead with that condition" rather than "state that condition", matching what today's
position experiment (`scripts/28...ipynb`, section 9) showed actually moves behaviour,
though that experiment only tested it on `gpt-4o-mini`, not these candidates.


In [2]:
COMPACT_SYSTEM_PROMPT = """You are a legal information assistant for Jordanian labour and social security law (the Labour Law 1996, the Professional Work Law 2019, and the Social Security Law). Answer using only the provided excerpts.

Grounding
1. Every statement must come from the documents -- no outside knowledge.
2. If the documents don't answer the question, say so plainly rather than filling the gap.
3. Cite by law and article number (e.g. "Labour Law 1996, Article 61"), only articles you were given, under the law each document is actually labelled with.

Coverage
4. Read every document before answering. More than one article often bears on the question -- address each one that does, with its own citation, not just the first.
5. Within a cited article, cover every lettered paragraph that adds a condition, exception, limit, minimum, or further step.
6. State each provision on its own terms. Don't explain how two provisions combine or override each other unless a document says so -- give each separately, side by side.

Precision
7. Name the specific paragraph that applies ("Article 65(a)"), never merging two paragraphs into one statement.
8. Reproduce durations, percentages, amounts, ages and deadlines exactly as given, attached to the specific obligation they belong to.
9. State any qualification together with the rule it qualifies. Never present a conditional rule as unconditional or widen it beyond the persons/cases named.

Usefulness
10. Answer the person's actual situation first, in a sentence or two, then give supporting rules and citations. Don't open by restating the article.
11. If the answer depends on a fact the person hasn't given (age, years of service, marital status, a date, an amount, which law covers them, or sex where a provision sets separate figures) -- ask for it in one short question rather than guessing or covering every case. Never silently default to one case (e.g. male figures) when a provision states both. If the documents give a short, complete set of conditions instead, state them directly. Judge the current question on its own terms even if an earlier turn answered without the fact. Exception: if a computed block already gives both cases (rule 20), presenting both IS the answer -- don't also ask.
12. Reply in the question's language. Be plain and unpadded, but never drop a relevant provision to stay short.

Judgment questions
13. For a personal-decision question ("should I...", "is it worth it..."), lay out the documents' benefits and obligations/costs neutrally, side by side, so the person can weigh it. Don't recommend, and don't reason beyond what the documents state.

Totals and calculations
14. For a total, combined rate, or amount worked out from a stated figure, account for EVERY contributing provision, not just the most prominent one -- contributions especially span several articles, one per insurance type.
15. Convert any rate the documents write in words to a numeral before calculating (using the conversion aid) -- it counts the same as a digit rate.
16. Show each component on its own line with its article and insurance type (never drop the name). Then the total. Keep employer and employee amounts separate; don't combine them unless asked for the combined total.
17. Apply any stated ceiling or floor, and say plainly when it doesn't affect the figure.
18. Adding up rates the documents state outright is arithmetic, not the inference rule 6 forbids -- rule 6 still governs everything else. If the total depends on which category the person falls into, say which case you calculated and name the others.
19. A computed block's arithmetic is already done and checked -- use its rates, amounts and totals exactly, without re-deriving, re-rounding, or contradicting them. It already satisfies rules 14-17; you still cite each rate's article and add anything the documents say that the block doesn't cover.
20. Follow a block's sex-differentiated condition exactly: one figure means sex is known (state only that one); both figures mean it isn't (present both, labelled) -- this has already been determined, not something to re-decide.

Multiple laws, and who a law covers
21. If a document states a condition that decides which of two provisions applies (most often, coverage by a different law), lead with that condition -- this is reading the text, not the inference rule 6 forbids. When two laws address the same event, never merge them into one undifferentiated answer: attribute each obligation to the party the text names, and never describe one law's institution's payment (e.g. Social Security) as required from the other's party (e.g. the employer).
22. Labour Law 1996 excludes government/municipal employees and domestic/agricultural workers (separate rules apply). Professional Work Law 2019 only covers professions the Council of Ministers has included. Treat a retrieved document's own coverage conditions the same way. Mention whichever is relevant briefly, once -- not every turn, and not as a second full answer; if these documents don't cover the person's situation, say so per rule 2.

Close with one short line: general information, not legal advice, consult a licensed lawyer about their own case."""

CONTEXT_SUFFIX = CONTEXT_SYSTEM_PROMPT[len(SYSTEM_PROMPT):]  # the citation-format paragraph, unchanged
COMPACT_CONTEXT_SYSTEM_PROMPT = COMPACT_SYSTEM_PROMPT + CONTEXT_SUFFIX

print("compact SYSTEM_PROMPT:", len(COMPACT_SYSTEM_PROMPT), "chars  "
      f"({1 - len(COMPACT_SYSTEM_PROMPT)/len(SYSTEM_PROMPT):.0%} shorter)")


compact SYSTEM_PROMPT: 5189 chars  (34% shorter)


## 2. Smoke test -- cheap sanity check before the full run

Three known-tricky cases across all three models with the compact prompt, before
spending on 180 calls: q05 (the two-laws case, rule 21), c01 (contributions
arithmetic, rules 14-19), c05 (gender-conditioned pension, rules 11/20). If any of
these obviously breaks -- e.g. the calculator block gets re-derived, or a gender
question silently defaults -- fix the compact prompt before the full run, not after.


In [3]:
retriever = Retriever()

ev_v3 = pd.read_json(EVAL_DIR / "eval_v3.jsonl", lines=True)
ev_calc = pd.read_json(EVAL_DIR / "eval_calc_v4.jsonl", lines=True)

smoke_questions = {
    "q05": ev_v3.loc[ev_v3.question_id == "q05", "question_text"].iloc[0],
    "c01": ev_calc.loc[ev_calc.question_id == "c01", "question_text"].iloc[0],
    "c05": ev_calc.loc[ev_calc.question_id == "c05", "question_text"].iloc[0],
}
smoke_context = {qid: retriever.retrieve(q, top_k=5) for qid, q in smoke_questions.items()}
for qid, chunks in smoke_context.items():
    print(qid, "->", [c["article_number"] for c in chunks])


q05 -> ['91', '90', '30', '96', '92', '86', '4']
c01 -> ['59', '42', '24', '48', '20', '53', '89', '90', '85', '83', '67', '72', '4']
c05 -> ['64', '62', '89', '50', '66', '4']


In [4]:
SMOKE_MODELS = {
    "gpt-4o-mini": PRODUCTION_MODEL_ID,
    "gemini-2.5-flash": "google/gemini-2.5-flash",
    "gpt-4o": "openai/gpt-4o",
}

smoke_rows = []
for label, model_id in SMOKE_MODELS.items():
    gen = OpenAICompatibleGenerator(model_id=model_id, api_key_env=PRODUCTION_API_KEY_ENV,
                                     base_url=PRODUCTION_BASE_URL, system_prompt=COMPACT_CONTEXT_SYSTEM_PROMPT)
    for qid, q in smoke_questions.items():
        res = gen.generate(q, smoke_context[qid])
        smoke_rows.append({"model": label, "question_id": qid, "answer": res["answer"],
                            "calculation": res.get("calculation")})
        print(f"--- {label} / {qid} (calc fired: {bool(res.get('calculation'))}) ---")
        print(res["answer"][:400])
        print()

smoke_df = pd.DataFrame(smoke_rows)
smoke_df.to_csv(EVAL_DIR / "compact_prompt_smoke_test.csv", index=False, encoding="utf-8-sig")


--- gpt-4o-mini / q05 (calc fired: False) ---
If someone is killed or left permanently unable to work due to a work accident, the compensation is as follows:

1. **For Death or Total Permanent Disability:**
   - The employer must pay compensation equivalent to the wages of one thousand and two hundred working days, but this compensation cannot exceed five thousand JDs and is not less than two thousand JDs. 
   (Labour Law 1996, Article 90(a))



--- gpt-4o-mini / c01 (calc fired: True) ---
يدفع صاحب العمل عنك للضمان الاجتماعي بمجموع قدره 142.5 دينار، وهذا يشمل الاشتراكات التالية:

- تأمين إصابات العمل: 20 دينار (قانون الضمان الاجتماعي، المادة 24).
- تأمين الأمومة: 7.5 دينار (قانون الضمان الاجتماعي، المادة 42).
- تأمين التعطل عن العمل: 5 دينار (قانون الضمان الاجتماعي، المادة 48).
- تأمين الشيخوخة والعجز والوفاة: 110 دينار (قانون الضمان الاجتماعي، المادة 59).

إجمالي المبلغ الذي يدفعه



--- gpt-4o-mini / c05 (calc fired: True) ---
لديك 200 اشتراك في الضمان الاجتماعي، وبالتالي تحتاج إلى الاشتراك في 52 اشتراك إضافي إذا كنت ذكراً (المجموع المطلوب 252 اشتراك). إذا كنت أنثى، تحتاج إلى 28 اشتراك إضافي (المجموع المطلوب 228 اشتراك).

- **إذا كان ذكراً:** تحتاج إلى 52 اشتراك إضافي حتى تصل إلى 252. 
- **إذا كانت أنثى:** تحتاج إلى 28 اشتراك إضافي حتى تصل إلى 228.

**الدعم القانوني:**
- للمؤمن عليه الذكر، يجب أن تكون اشتراكاته في هذا ا



--- gemini-2.5-flash / q05 (calc fired: False) ---
If a worker dies or suffers total permanent disability due to a work injury, there are provisions for compensation under both the Labor Law 1996 and the Social Security Law, depending on whether the worker is subject to the Social Security Law.

**Under the Labor Law 1996:**

*   If the work injury results in the death or total disability of the worker, the employer is liable to pay compensation e



--- gemini-2.5-flash / c01 (calc fired: True) ---
يدفع صاحب العمل 142.5 دينار عنك للضمان الاجتماعي، وذلك باحتساب 14.25% من راتبك الشهري البالغ 1000 دينار.

تتوزع هذه الاشتراكات كالتالي:
*   تأمين إصابات العمل: 20 دينار (2% من الأجر) (قانون الضمان الاجتماعي، المادة 24(أ))
*   تأمين الأمومة: 7.5 دينار (0.75% من الأجر) (قانون الضمان الاجتماعي، المادة 42(أ)1)
*   تأمين التعطل عن العمل: 5 دينار (0.5% من الأجر) (قانون الضمان الاجتماعي، المادة 48(أ))
* 



--- gemini-2.5-flash / c05 (calc fired: True) ---
للتقاعد المبكر، يتطلب قانون الضمان الاجتماعي الأعداد التالية من الاشتراكات:

*   إذا كنت ذكراً، يلزمك 252 اشتراكاً فعلياً على الأقل. لديك 200 اشتراك حاليًا، مما يعني أنك بحاجة إلى 52 اشتراكاً إضافياً. (قانون الضمان الاجتماعي، المادة 64(أ)(1))
*   إذا كنت أنثى، يلزمك 228 اشتراكاً فعلياً على الأقل. لديك 200 اشتراك حاليًا، مما يعني أنك بحاجة إلى 28 اشتراكاً إضافياً. (قانون الضمان الاجتماعي، المادة 64



--- gpt-4o / q05 (calc fired: False) ---
If a worker is killed or left permanently unable to work due to a work injury, payments and amounts are determined by two different laws depending on their coverage situation. Here's a breakdown:

Workers not covered by the Social Security Law (those covered by Labor Law 1996 only):

1. **Death or Total Disability:** 
   - Compensation equal to 1,200 working days' wages, subject to a ceiling of 5,



--- gpt-4o / c01 (calc fired: True) ---
يدفع صاحب العمل عنك للضمان الاجتماعي مبلغ إجمالي قدره 142.5 دينار شهرياً على راتب مقداره 1000 دينار. التفاصيل كالتالي:

- "تأمين إصابات العمل" يؤدى بنسبة 2% = 20 دينار. (قانون الضمان الاجتماعي، المادة 24)
- "تأمين الأمومة" يؤدى بنسبة 0.75% = 7.5 دينار. (قانون الضمان الاجتماعي، المادة 42)
- "تأمين التعطل عن العمل" يؤدى بنسبة 0.5% = 5 دينار. (قانون الضمان الاجتماعي، المادة 48)
- "تأمين الشيخوخة والع



--- gpt-4o / c05 (calc fired: True) ---
للتقاعد المبكر حسب "قانون الضمان الاجتماعي، المادة (64)"، تحتاج إلى تحقيق شروط معينة تختلف بناءً على الجنس:

إذا كنت ذكراً، يجب أن تكون قد أكملت 252 اشتراكاً فعلياً. لديك حالياً 200 اشتراك، لذا تحتاج إلى 52 اشتراكاً إضافياً.

إذا كنت أنثى، يجب أن تكون قد أكملت 228 اشتراكاً فعلياً. لديك حالياً 200 اشتراك، لذا تحتاج إلى 28 اشتراكات إضافياً.

يرجى استشارة محامٍ مرخص للحصول على معلومات قانونية خاصة بح



**Read the 9 answers above before continuing.** Check specifically: does c01/c05 still
show the calculator block firing (`calc fired: True`) and does the narration match it
without re-deriving numbers; does q05 still blend the two laws or does the tightened
rule 21 wording help on the candidate models (not expected to help gpt-4o-mini, per
`scripts/28...ipynb`). If anything looks broken, stop and fix `COMPACT_SYSTEM_PROMPT`
before running the cell below.


## 3. Full re-review -- all 60 `eval_v3` questions

Production baseline uses its real shipped prompt (`CONTEXT_SYSTEM_PROMPT`, unmodified).
Both candidates use `COMPACT_CONTEXT_SYSTEM_PROMPT`. Context is retrieved once per
question and reused across all three arms (paired, per Phase 21's principle) --
today's retrieval, including Phase 28's uncommitted precondition-article mechanism.


In [5]:
RETRIEVAL_PATH = EVAL_DIR / "rereview_retrieval.csv"
GEN_PATH = EVAL_DIR / "rereview_generations.csv"
JUDGE_PATH = EVAL_DIR / "rereview_judge.csv"

ev60 = pd.read_json(EVAL_DIR / "eval_v3.jsonl", lines=True)
assert len(ev60) == 60

# Context isn't cached to disk (not cleanly serialisable) -- re-retrieved fresh each run,
# which is cheap (60 embedding calls) relative to the generation/judge cost below.
context_by_qid = {}
retr_rows = []
for q in ev60.itertuples(index=False):
    chunks = retriever.retrieve(q.question_text, top_k=5)
    context_by_qid[q.question_id] = chunks
    gold = set(q.answer_article_numbers)
    retr_rows.append({
        "question_id": q.question_id,
        "gold_articles": sorted(gold),
        "retrieved_articles": [c["article_number"] for c in chunks],
        "full_coverage": gold.issubset({c["article_number"] for c in chunks}),
    })

retr_df = pd.DataFrame(retr_rows)
retr_df.to_csv(RETRIEVAL_PATH, index=False, encoding="utf-8-sig")
print(f"full coverage: {retr_df.full_coverage.sum()}/{len(retr_df)}")


full coverage: 58/60


In [6]:
ARMS = {
    "gpt-4o-mini (production)": (PRODUCTION_MODEL_ID, None),               # real shipped prompt
    "gemini-2.5-flash (compact)": ("google/gemini-2.5-flash", COMPACT_CONTEXT_SYSTEM_PROMPT),
    "gpt-4o (compact)": ("openai/gpt-4o", COMPACT_CONTEXT_SYSTEM_PROMPT),
}

gen_rows = []
if GEN_PATH.exists():
    gen_rows = pd.read_csv(GEN_PATH, encoding="utf-8-sig").to_dict("records")
done = {(r["arm"], r["question_id"]) for r in gen_rows}
print(f"{len(done)} already generated")

generators = {arm: OpenAICompatibleGenerator(model_id=mid, api_key_env=PRODUCTION_API_KEY_ENV,
                                              base_url=PRODUCTION_BASE_URL, system_prompt=sp)
              for arm, (mid, sp) in ARMS.items()}

for arm in ARMS:
    gen = generators[arm]
    for q in ev60.itertuples(index=False):
        if (arm, q.question_id) in done:
            continue
        chunks = context_by_qid[q.question_id]
        res = gen.generate(q.question_text, chunks)
        gen_rows.append({
            "arm": arm, "question_id": q.question_id, "question_text": q.question_text,
            "question_language": q.question_language, "answer": res["answer"],
            "citations": res.get("citations"), "calculation": res.get("calculation"),
        })
        pd.DataFrame(gen_rows).to_csv(GEN_PATH, index=False, encoding="utf-8-sig")
        print(f"  {arm} / {q.question_id}", end="\r")

gen_df = pd.DataFrame(gen_rows)
print(f"\n{len(gen_df)} generations total (expect {len(ARMS) * len(ev60)})")


0 already generated


  gpt-4o (compact) / q60
180 generations total (expect 180)


## 4. Regex proxy

In [7]:
gold_by_qid = {q.question_id: set(q.answer_article_numbers) for q in ev60.itertuples(index=False)}

proxy_rows = []
for r in gen_df.itertuples(index=False):
    cited = extract_cited_articles(r.answer)
    gold = gold_by_qid[r.question_id]
    grounded = grounded_article_numbers(context_by_qid[r.question_id])
    proxy_rows.append({
        "arm": r.arm, "question_id": r.question_id,
        "correct_article_cited": bool(cited & gold),
        "all_correct_articles_cited": gold.issubset(cited),
        "has_fabrication": bool(cited - grounded),
    })

proxy_df = pd.DataFrame(proxy_rows)
proxy_df.groupby("arm")[["correct_article_cited", "all_correct_articles_cited", "has_fabrication"]].agg(
    lambda s: f"{s.sum()}/{len(s)}")


,correct_article_cited,all_correct_articles_cited,has_fabrication
arm,,,
gemini-2.5-flash (compact),60/60,58/60,1/60
gpt-4o (compact),58/60,55/60,0/60
gpt-4o-mini (production),58/60,55/60,1/60


## 5. LLM judge -- Claude Sonnet 5, all 180 calls, incremental persistence

In [8]:
from evallib import make_judge

JUDGE_MAX_TOKENS = 2000
judge_client = OpenAI(api_key=os.environ[PRODUCTION_API_KEY_ENV], base_url=PRODUCTION_BASE_URL)
judge_one = make_judge(judge_client, max_tokens=JUDGE_MAX_TOKENS)

judge_rows = []
if JUDGE_PATH.exists():
    judge_rows = pd.read_csv(JUDGE_PATH, encoding="utf-8-sig").to_dict("records")
jdone = {(r["arm"], r["question_id"]) for r in judge_rows}
print(f"{len(jdone)} already judged, {len(gen_df) - len(jdone)} to go")

for r in gen_df.itertuples(index=False):
    if (r.arm, r.question_id) in jdone:
        continue
    verdict = judge_one(r.question_text, context_by_qid[r.question_id], r.answer)
    judge_rows.append({"arm": r.arm, "question_id": r.question_id, **verdict})
    pd.DataFrame(judge_rows).to_csv(JUDGE_PATH, index=False, encoding="utf-8-sig")
    print(f"  judged {len(judge_rows)}/{len(gen_df)}", end="\r")

judge_df = pd.DataFrame(judge_rows)
print(f"\n{len(judge_df)} judged")


0 already judged, 180 to go


  judged 180/180
180 judged


## 6. Results

In [9]:
print("=" * 72)
print("RETRIEVAL (identical across all arms)")
print("=" * 72)
print(f"  full coverage  {retr_df.full_coverage.sum()}/{len(retr_df)}")

print()
print("=" * 72)
print("GENERATION -- regex proxy")
print("=" * 72)
print(proxy_df.groupby("arm")[["correct_article_cited", "all_correct_articles_cited", "has_fabrication"]]
      .agg(lambda s: f"{s.sum()}/{len(s)}"))

print()
print("=" * 72)
print("GENERATION -- LLM judge (Claude Sonnet 5, 1-5)")
print("=" * 72)
print(judge_df.groupby("arm")[["faithfulness_score", "relevancy_score"]].agg(["mean", "median", "min"]))


RETRIEVAL (identical across all arms)
  full coverage  58/60

GENERATION -- regex proxy
                           correct_article_cited all_correct_articles_cited  \
arm                                                                           
gemini-2.5-flash (compact)                 60/60                      58/60   
gpt-4o (compact)                           58/60                      55/60   
gpt-4o-mini (production)                   58/60                      55/60   

                           has_fabrication  
arm                                         
gemini-2.5-flash (compact)            1/60  
gpt-4o (compact)                      0/60  
gpt-4o-mini (production)              1/60  

GENERATION -- LLM judge (Claude Sonnet 5, 1-5)
                           faithfulness_score            relevancy_score  \
                                         mean median min            mean   
arm                                                                        
gemini-2.5-flas

In [10]:
from math import comb


def binom_two_sided_p(k, n, p=0.5):
    """Exact two-sided sign-test p-value, no scipy dependency (not installed in rag311)."""
    if n == 0:
        return float("nan")
    pmf = [comb(n, i) * p**i * (1 - p)**(n - i) for i in range(n + 1)]
    pk = pmf[k]
    return min(1.0, sum(x for x in pmf if x <= pk + 1e-12))


baseline = "gpt-4o-mini (production)"
prod_scores = judge_df[judge_df.arm == baseline].set_index("question_id")

print("Paired vs production, ties excluded (sign test):")
for arm in ARMS:
    if arm == baseline:
        continue
    cand = judge_df[judge_df.arm == arm].set_index("question_id")
    for metric in ["faithfulness_score", "relevancy_score"]:
        diff = cand[metric] - prod_scores.loc[cand.index, metric]
        wins = int((diff > 0).sum())
        losses = int((diff < 0).sum())
        n = wins + losses
        p = binom_two_sided_p(wins, n, 0.5) if n else float("nan")
        print(f"  {arm:28s} {metric:20s} {wins}-{losses} (ties={int((diff==0).sum())})  p={p:.3f}  "
              f"mean diff={diff.mean():+.3f}")


Paired vs production, ties excluded (sign test):
  gemini-2.5-flash (compact)   faithfulness_score   14-8 (ties=38)  p=0.286  mean diff=+0.200
  gemini-2.5-flash (compact)   relevancy_score      11-22 (ties=27)  p=0.080  mean diff=-0.300
  gpt-4o (compact)             faithfulness_score   15-6 (ties=39)  p=0.078  mean diff=+0.200
  gpt-4o (compact)             relevancy_score      9-7 (ties=44)  p=0.804  mean diff=+0.000


## Verdict

**Against the pre-committed rule (beat production on both faithfulness and relevancy, paired, by a real margin): neither candidate clears it. Nothing is adopted.**

| candidate | faithfulness vs production | relevancy vs production |
|---|---|---|
| gemini-2.5-flash (compact) | 14-8, p=0.286, +0.200 -- not significant | **11-22, p=0.080, -0.300 -- a LOSS** |
| gpt-4o (compact) | 15-6, p=0.078, +0.200 -- closest to significant, not quite | 9-7, p=0.804, +0.000 -- flat tie |

`gemini-2.5-flash` fails outright: it loses on relevancy, the opposite of what adoption requires. `gpt-4o` doesn't fail as clearly -- it's directionally better on faithfulness and never worse on relevancy -- but "directionally better, not significant" is exactly the case Phase 15 already established this project should not talk itself into overriding a pre-committed rule for. Per that standard: **DECISION UNRESOLVED, NOTHING CHANGED.** `src/generation.py` stays on `gpt-4o-mini` with the original (uncompacted) prompt.

**Why gemini lost on relevancy, read from the judge's own reasons, not guessed:** every one of gemini's 22 relevancy losses that were read say the same thing -- "correct, but padded with tangential detail not asked for" (q22: padded with articles 45/56/47 detail; q20: diluted with unrelated military procedure; q33: padded with dismissal-exception detail; q40: padded with juvenile-worker and social-security detail). This is measurable, not just judge impression: **mean answer length gemini 1,586 chars vs gpt-4o 727 vs production 804** -- gemini's answers are roughly twice as long on average, despite the compact prompt's rule 12 ("write plainly and without padding") being sent to all three. The coverage rules (4-5) were tuned in Phase 21 specifically because `gpt-4o-mini` was *under*-covering; gemini doesn't have that problem and the same instruction pushes it into *over*-covering instead. **One of gemini's worst losses (q13, -3) is a separate, real bug, not padding:** the answer is cut off mid-word ("You also cannot combine unemployment") -- a genuine truncation, most likely `OpenAICompatibleGenerator.generate()` never sets `max_tokens` explicitly and Gemini's OpenRouter default behaves differently under that than OpenAI's does. This needs fixing (explicit `max_tokens`) before gemini could be fairly re-tested at all -- today's number is measured with that bug live.

**`gpt-4o` is the more promising candidate, close but not proven:** its compact-prompt answers are actually *shorter* than production's original-prompt answers (727 vs 804 mean chars) -- the compact prompt's conciseness push worked on this model where it didn't on gemini -- and it never lost on relevancy. Its faithfulness edge (p=0.078) is real in direction but this run alone can't call it significant.

**Proxy and retrieval, for the record (not decision-driving here, per this project's separate-metrics rule):** retrieval full coverage 58/60 (matches Phase 27's number, no regression). Regex proxy: gemini 60/60 correct-article-cited (best of the three), gpt-4o and production tied 58/60; all-correct-articles gemini 58/60 vs 55/60 for the other two. Fabrication flags near-zero for all three (0-1/60), consistent with this project's established finding that these flags are almost always the in-text cross-reference false positive, not real fabrication.

**Scope caveat restated:** this run tested "new model + compact prompt" as one bundled unit against "old model + old prompt" -- a win or loss here cannot be attributed to the model alone.

**If this is worth continuing rather than closing:** (1) fix the Gemini truncation bug (explicit `max_tokens`) and re-run gemini only -- its relevancy number is not trustworthy until that's fixed; (2) a second `gpt-4o` run (different day, or a paired split-half per Phase 18's methodology) would say whether its near-significant faithfulness edge is real or this run's particular noise; (3) neither is a small ask -- both cost real money again, and per the standing rule in this project, that needs a fresh go-ahead, not an assumption.
